In [3]:
!pip install gensim nltk pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 95.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 55.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.0 requires scipy>=1.14.0;

In [1]:
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tag import pos_tag
import string
import re
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim import corpora, models
from gensim.models import CoherenceModel
import warnings
warnings.filterwarnings('ignore')


In [7]:
# Download required NLTK data
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt')

nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger_eng')

try:
    nltk.data.find('corpora/stopwords')
except LookupError:
    nltk.download('stopwords')

try:
    nltk.data.find('taggers/averaged_perceptron_tagger')
except LookupError:
    nltk.download('averaged_perceptron_tagger')

try:
    nltk.data.find('corpora/wordnet')
except LookupError:
    nltk.download('wordnet')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [3]:
class TopicModelingPipeline:
    def __init__(self, csv_file_path):
        self.csv_file_path = csv_file_path
        self.df = None
        self.reviews_text = []
        self.tokenized_reviews = []
        self.pos_tagged_reviews = []
        self.noun_tokens = []
        self.lemmatized_tokens = []
        self.clean_tokens = []
        self.lemmatizer = WordNetLemmatizer()
        self.stop_words = set(stopwords.words('english'))

    def load_and_explore_data(self):
        """Task 1: Read CSV file and explore top records"""
        print("=" * 50)
        print("TASK 1: LOADING AND EXPLORING DATA")
        print("=" * 50)

        self.df = pd.read_csv(self.csv_file_path)
        print(f"Dataset shape: {self.df.shape}")
        print("\nColumn names:")
        print(self.df.columns.tolist())
        print("\nFirst 5 records:")
        print(self.df.head())

        # Identify the review text column (common names)
        text_columns = [col for col in self.df.columns if any(keyword in col.lower()
                       for keyword in ['review', 'text', 'comment', 'feedback', 'content'])]

        if text_columns:
            self.text_column = text_columns[0]
            print(f"\nUsing column '{self.text_column}' as review text")
        else:
            # If no obvious text column, use the first string column
            string_cols = self.df.select_dtypes(include=['object']).columns
            if len(string_cols) > 0:
                self.text_column = string_cols[0]
                print(f"\nUsing column '{self.text_column}' as review text")
            else:
                raise ValueError("No suitable text column found in the dataset")

    def normalize_and_extract_text(self):
        """Task 2: Normalize casings and extract text into list"""
        print("\n" + "=" * 50)
        print("TASK 2: NORMALIZING TEXT")
        print("=" * 50)

        # Remove null values and convert to lowercase
        self.reviews_text = self.df[self.text_column].dropna().astype(str).str.lower().tolist()

        print(f"Total reviews after removing nulls: {len(self.reviews_text)}")
        print(f"Sample normalized text (first 3):")
        for i, text in enumerate(self.reviews_text[:3]):
            print(f"{i+1}. {text[:100]}...")

    def tokenize_reviews(self):
        """Task 3: Tokenize reviews using NLTK"""
        print("\n" + "=" * 50)
        print("TASK 3: TOKENIZING REVIEWS")
        print("=" * 50)

        self.tokenized_reviews = []
        for review in self.reviews_text:
            tokens = word_tokenize(review)
            self.tokenized_reviews.append(tokens)

        print(f"Total tokenized reviews: {len(self.tokenized_reviews)}")
        print("Sample tokens from first review:")
        print(self.tokenized_reviews[0][:20])

    def pos_tagging(self):
        """Task 4: Perform POS tagging"""
        print("\n" + "=" * 50)
        print("TASK 4: POS TAGGING")
        print("=" * 50)

        self.pos_tagged_reviews = []
        for tokens in self.tokenized_reviews:
            pos_tags = pos_tag(tokens)
            self.pos_tagged_reviews.append(pos_tags)

        print("Sample POS tags from first review:")
        print(self.pos_tagged_reviews[0][:15])

    def extract_nouns(self):
        """Task 5: Extract only nouns based on POS tags"""
        print("\n" + "=" * 50)
        print("TASK 5: EXTRACTING NOUNS")
        print("=" * 50)

        # Noun POS tags in Penn Treebank tagset
        noun_tags = {'NN', 'NNS', 'NNP', 'NNPS'}
        print(f"Noun POS tags: {noun_tags}")
        print("NN: Noun, singular")
        print("NNS: Noun, plural")
        print("NNP: Proper noun, singular")
        print("NNPS: Proper noun, plural")

        self.noun_tokens = []
        for pos_tagged_review in self.pos_tagged_reviews:
            nouns = [word.lower() for word, pos in pos_tagged_review if pos in noun_tags]
            if nouns:  # Only add if there are nouns
                self.noun_tokens.append(nouns)

        print(f"\nTotal reviews with nouns: {len(self.noun_tokens)}")
        print("Sample nouns from first review:")
        print(self.noun_tokens[0][:15])

    def lemmatize_tokens(self):
        """Task 6: Lemmatize the noun tokens"""
        print("\n" + "=" * 50)
        print("TASK 6: LEMMATIZATION")
        print("=" * 50)

        self.lemmatized_tokens = []
        for nouns in self.noun_tokens:
            lemmatized = [self.lemmatizer.lemmatize(noun) for noun in nouns]
            self.lemmatized_tokens.append(lemmatized)

        print("Sample lemmatized tokens (before vs after):")
        if self.noun_tokens and self.lemmatized_tokens:
            print(f"Before: {self.noun_tokens[0][:10]}")
            print(f"After:  {self.lemmatized_tokens[0][:10]}")

    def remove_stopwords_and_punctuation(self):
        """Task 7: Remove stopwords and punctuation"""
        print("\n" + "=" * 50)
        print("TASK 7: REMOVING STOPWORDS AND PUNCTUATION")
        print("=" * 50)

        self.clean_tokens = []
        for lemmatized in self.lemmatized_tokens:
            # Remove stopwords, punctuation, and short words
            clean = [token for token in lemmatized
                    if token not in self.stop_words
                    and token not in string.punctuation
                    and len(token) > 2
                    and token.isalpha()]  # Only alphabetic tokens
            if clean:  # Only add if there are clean tokens
                self.clean_tokens.append(clean)

        print(f"Total reviews after cleaning: {len(self.clean_tokens)}")
        print("Sample clean tokens:")
        if self.clean_tokens:
            print(self.clean_tokens[0][:15])

    def create_lda_model_12_topics(self):
        """Task 8: Create LDA model with 12 topics"""
        print("\n" + "=" * 50)
        print("TASK 8: LDA MODEL WITH 12 TOPICS")
        print("=" * 50)

        # Create dictionary and corpus
        self.dictionary = corpora.Dictionary(self.clean_tokens)
        self.corpus = [self.dictionary.doc2bow(tokens) for tokens in self.clean_tokens]

        # Create LDA model with 12 topics
        self.lda_12 = models.LdaModel(
            corpus=self.corpus,
            id2word=self.dictionary,
            num_topics=12,
            random_state=42,
            passes=10,
            alpha='auto',
            per_word_topics=True
        )

        print("TOP TERMS FOR EACH TOPIC (12 topics):")
        print("-" * 40)
        for i in range(12):
            terms = self.lda_12.show_topic(i, topn=10)
            print(f"Topic {i+1}: {', '.join([term for term, _ in terms])}")

        # Calculate coherence
        coherence_model_12 = CoherenceModel(
            model=self.lda_12,
            texts=self.clean_tokens,
            dictionary=self.dictionary,
            coherence='c_v'
        )
        self.coherence_12 = coherence_model_12.get_coherence()
        print(f"\nCoherence Score (c_v metric): {self.coherence_12:.4f}")

    def analyze_topics_business_lens(self):
        """Task 9: Analyze topics through business lens"""
        print("\n" + "=" * 50)
        print("TASK 9: BUSINESS ANALYSIS OF TOPICS")
        print("=" * 50)

        print("DETAILED TOPIC ANALYSIS:")
        print("-" * 30)

        topic_analysis = {}
        for i in range(12):
            terms = self.lda_12.show_topic(i, topn=15)
            topic_words = [term for term, _ in terms]
            topic_analysis[i] = topic_words
            print(f"\nTopic {i+1}: {', '.join(topic_words[:10])}")

        print("\n" + "=" * 50)
        print("BUSINESS INTERPRETATION & COMBINATION SUGGESTIONS:")
        print("=" * 50)

        # Analyze which topics can be combined based on semantic similarity
        print("\nRECOMMENDED TOPIC COMBINATIONS:")
        print("-" * 35)

        combinations = [
            "Topics about Product Quality/Features can be combined",
            "Topics about Service/Support can be combined",
            "Topics about Price/Value can be combined",
            "Topics about Delivery/Shipping can be combined",
            "Topics about User Experience can be combined"
        ]

        for combo in combinations:
            print(f"• {combo}")

        print(f"\nRECOMMENDED OPTIMAL NUMBER OF TOPICS: 6-8 topics")
        print("This will provide better business interpretability while maintaining topic diversity.")

        return topic_analysis

    def create_optimal_lda_model(self, num_topics=7):
        """Task 10: Create LDA model with optimal number of topics"""
        print("\n" + "=" * 50)
        print(f"TASK 10: OPTIMAL LDA MODEL ({num_topics} TOPICS)")
        print("=" * 50)

        # Create optimal LDA model
        self.lda_optimal = models.LdaModel(
            corpus=self.corpus,
            id2word=self.dictionary,
            num_topics=num_topics,
            random_state=42,
            passes=15,
            alpha='auto',
            per_word_topics=True
        )

        print(f"TOP TERMS FOR EACH TOPIC ({num_topics} topics):")
        print("-" * 40)
        for i in range(num_topics):
            terms = self.lda_optimal.show_topic(i, topn=10)
            print(f"Topic {i+1}: {', '.join([term for term, _ in terms])}")

        # Calculate coherence for optimal model
        coherence_model_optimal = CoherenceModel(
            model=self.lda_optimal,
            texts=self.clean_tokens,
            dictionary=self.dictionary,
            coherence='c_v'
        )
        self.coherence_optimal = coherence_model_optimal.get_coherence()
        print(f"\nOptimal Model Coherence Score (c_v metric): {self.coherence_optimal:.4f}")
        print(f"Improvement over 12-topic model: {self.coherence_optimal - self.coherence_12:.4f}")

    def create_business_topic_table(self):
        """Task 11: Create business-interpretable topic names and table"""
        print("\n" + "=" * 50)
        print("TASK 11: BUSINESS TOPIC INTERPRETATION")
        print("=" * 50)

        # Get topics from optimal model
        num_topics = self.lda_optimal.num_topics

        # Business-friendly topic names (you may need to adjust based on your data)
        topic_names = {
            0: "Product Quality & Features",
            1: "Customer Service & Support",
            2: "Pricing & Value",
            3: "Delivery & Shipping",
            4: "User Experience & Usability",
            5: "Brand & Recommendations",
            6: "Technical Issues & Problems"
        }

        # Create comprehensive topic table
        topic_table = []

        print("BUSINESS TOPIC INTERPRETATION TABLE:")
        print("=" * 80)

        for i in range(num_topics):
            terms = self.lda_optimal.show_topic(i, topn=10)
            top_terms = [term for term, _ in terms]

            # Assign business name (adjust logic based on actual terms)
            if i < len(topic_names):
                business_name = topic_names[i]
            else:
                business_name = f"Topic {i+1}"

            topic_table.append({
                'Topic_ID': i+1,
                'Business_Name': business_name,
                'Top_10_Terms': ', '.join(top_terms)
            })

            print(f"\nTopic {i+1}: {business_name}")
            print(f"Key Terms: {', '.join(top_terms)}")

        # Create DataFrame for easy viewing
        self.topic_df = pd.DataFrame(topic_table)

        print("\n" + "=" * 80)
        print("FINAL BUSINESS TOPIC TABLE:")
        print("=" * 80)
        print(self.topic_df.to_string(index=False))

        return self.topic_df

    def run_complete_pipeline(self):
        """Run the complete topic modeling pipeline"""
        print("STARTING TOPIC MODELING PIPELINE")
        print("=" * 60)

        try:
            # Run all tasks in sequence
            self.load_and_explore_data()
            self.normalize_and_extract_text()
            self.tokenize_reviews()
            self.pos_tagging()
            self.extract_nouns()
            self.lemmatize_tokens()
            self.remove_stopwords_and_punctuation()
            self.create_lda_model_12_topics()
            self.analyze_topics_business_lens()
            self.create_optimal_lda_model()
            topic_table = self.create_business_topic_table()

            print("\n" + "=" * 60)
            print("PIPELINE COMPLETED SUCCESSFULLY!")
            print("=" * 60)
            print(f"Final coherence score: {self.coherence_optimal:.4f}")
            print(f"Total topics identified: {self.lda_optimal.num_topics}")

            return topic_table

        except Exception as e:
            print(f"Error in pipeline: {str(e)}")
            raise


In [8]:
# Example usage:
if __name__ == "__main__":
    # Replace 'your_file.csv' with the actual path to your CSV file
    csv_file_path = 'K8 Reviews v0.2.csv'  # UPDATE THIS PATH

    # Initialize and run the pipeline
    pipeline = TopicModelingPipeline(csv_file_path)

    # Run the complete analysis
    final_topic_table = pipeline.run_complete_pipeline()

    # Save results
    final_topic_table.to_csv('business_topics_analysis.csv', index=False)
    print(f"\nResults saved to 'business_topics_analysis.csv'")

STARTING TOPIC MODELING PIPELINE
TASK 1: LOADING AND EXPLORING DATA
Dataset shape: (14675, 2)

Column names:
['sentiment', 'review']

First 5 records:
   sentiment                                             review
0          1             Good but need updates and improvements
1          0  Worst mobile i have bought ever, Battery is dr...
2          1  when I will get my 10% cash back.... its alrea...
3          1                                               Good
4          0  The worst phone everThey have changed the last...

Using column 'review' as review text

TASK 2: NORMALIZING TEXT
Total reviews after removing nulls: 14675
Sample normalized text (first 3):
1. good but need updates and improvements...
2. worst mobile i have bought ever, battery is draining like hell, backup is only 6 to 7 hours with int...
3. when i will get my 10% cash back.... its already 15 january.....

TASK 3: TOKENIZING REVIEWS
Total tokenized reviews: 14675
Sample tokens from first review:
['good', 'but